In [2]:
import numpy as np
import matplotlib.pyplot as plt
from stratified_box import StratifiedBox
import os
import astropy.units as u
import astropy.constants as C
from cooling import get_t_cool_cgs

In [3]:
noturb_sim_path = "/viper/ptmp/ferhi/StratDisk/Infall/chi1e3/longernoturb"
turb_sim_path = "/viper/ptmp/ferhi/StratDisk/Infall/chi1e3/longerturb_m0.03"
turb_sim = StratifiedBox(os.path.join(turb_sim_path, 'strat.in'), dir = turb_sim_path)
noturb_sim = StratifiedBox(os.path.join(noturb_sim_path, 'strat.in'), dir = noturb_sim_path)

noturb_hst_data = np.loadtxt(os.path.join(noturb_sim_path, 'out/parthenon.out1.hst'))
turb_hst_data = np.loadtxt(os.path.join(turb_sim_path, 'out/parthenon.out1.hst'))

this is tff:,  442.52862374439707


In [ ]:
noturb_vy = noturb_hst_data[:, 13] / noturb_hst_data[:, 10] * noturb_sim.code_length_cgs / noturb_sim.code_time_cgs
noturb_turbv = np.sqrt(noturb_hst_data[:, 11]**2 * noturb_hst_data[:, 12]**2 * noturb_hst_data[:, 13]**2) / noturb_hst_data[:, 10] * noturb_sim.code_length_cgs / noturb_sim.code_time_cgs
turb_vy = turb_hst_data[:, 13] / turb_hst_data[:, 10] * turb_sim.code_length_cgs / turb_sim.code_time_cgs


noturb_fall_time = noturb_hst_data[-1, 0] * noturb_sim.code_time_cgs / u.Myr.to('s')
turb_fall_time = (turb_hst_data[-1, 0] - turb_sim.t_inject)* turb_sim.code_time_cgs / u.Myr.to('s')
print("Noturb fall time (Myr): ", noturb_fall_time)
print("Turb fall time (Myr): ", turb_fall_time)

Noturb fall time (Myr):  2.73481
Turb fall time (Myr):  1.2041554547028792


/tmp/ipykernel_2078552/1093490266.py:3: RuntimeWarning: invalid value encountered in divide
  turb_vy = turb_hst_data[:, 13] / turb_hst_data[:, 10] * turb_sim.code_length_cgs / turb_sim.code_time_cgs


In [29]:
# Dimensions of simulation box
Lymin, Lymax = float(noturb_sim.reader.get('parthenon/mesh', 'x1min')), float(noturb_sim.reader.get('parthenon/mesh', 'x1max'))
H = (Lymax - Lymin) / 2 * noturb_sim.code_length_cgs


# Different timescales
t_fdrag = H / np.sqrt(2 * noturb_sim.chi * noturb_sim.r_cloud_inserted * noturb_sim.code_length_cgs * noturb_sim.g) / u.Myr.to('s')
t_ff = np.sqrt(2 * H / noturb_sim.g) / u.Myr.to('s')
t_fgrow = H / (noturb_sim.g * noturb_sim.chi * np.sqrt(noturb_sim.r_cloud_inserted * noturb_sim.code_length_cgs / (-noturb_vy) * get_t_cool_cgs(noturb_sim.cloud_rho, noturb_sim.T_cloud, noturb_sim.mbar))) / u.Myr.to('s')

print("t_fdrag (Myr): ", t_fdrag)
print("t_ff (Myr): ", t_ff)
print("t_fgrow (Myr): ", t_fgrow[-1])
print(noturb_vy)

t_fdrag (Myr):  0.10425239745068086
t_ff (Myr):  1.648375137388414
t_fgrow (Myr):  0.6845034606251625
[ 0.00000000e+00 -1.12450490e+02 -2.38833481e+02 ... -5.10857497e+05
 -5.09810605e+05 -5.09406294e+05]


/tmp/ipykernel_2078552/1834807533.py:9: RuntimeWarning: divide by zero encountered in divide
  t_fgrow = H / (noturb_sim.g * noturb_sim.chi * np.sqrt(noturb_sim.r_cloud_inserted * noturb_sim.code_length_cgs / (-noturb_vy) * get_t_cool_cgs(noturb_sim.cloud_rho, noturb_sim.T_cloud, noturb_sim.mbar))) / u.Myr.to('s')
/tmp/ipykernel_2078552/1834807533.py:9: RuntimeWarning: invalid value encountered in sqrt
  t_fgrow = H / (noturb_sim.g * noturb_sim.chi * np.sqrt(noturb_sim.r_cloud_inserted * noturb_sim.code_length_cgs / (-noturb_vy) * get_t_cool_cgs(noturb_sim.cloud_rho, noturb_sim.T_cloud, noturb_sim.mbar))) / u.Myr.to('s')


In [24]:
# Dimensions of simulation box
Lymin, Lymax = float(turb_sim.reader.get('parthenon/mesh', 'x1min')), float(turb_sim.reader.get('parthenon/mesh', 'x1max'))
H = (Lymax - Lymin) / 2 * turb_sim.code_length_cgs
cs = np.sqrt(C.k_B.cgs.value * turb_sim.T_base / (turb_sim.mbar))
print(0.03* cs/1e5)


# Different timescales
t_fdrag = H / np.sqrt(2 * turb_sim.chi * turb_sim.r_cloud_inserted * turb_sim.code_length_cgs * turb_sim.g) / u.Myr.to('s')
t_ff = np.sqrt(2 * H / turb_sim.g) / u.Myr.to('s')
t_fgrow = H / (turb_sim.g * turb_sim.chi * np.sqrt(turb_sim.r_cloud_inserted * turb_sim.code_length_cgs / (0.03 * cs) * get_t_cool_cgs(turb_sim.cloud_rho, turb_sim.T_cloud, turb_sim.mbar))) / u.Myr.to('s')

print("t_fdrag (Myr): ", t_fdrag)
print("t_ff (Myr): ", t_ff)
print("t_fgrow (Myr): ", t_fgrow)
print("t_fall,turb (Myr): ", H / (0.03 * cs) / u.Myr.to('s'))

11.234247273493798
t_fdrag (Myr):  0.10425239745068086
t_ff (Myr):  1.648375137388414
t_fgrow (Myr):  1.0165189614699097
t_fall,turb (Myr):  0.4177763359655952


In [33]:
get_t_cool_cgs(turb_sim.cloud_rho/100, turb_sim.T_cloud*100, turb_sim.mbar)/ get_t_cool_cgs(turb_sim.cloud_rho/10, turb_sim.T_cloud*10, turb_sim.mbar)

np.float64(305.4452201485708)